In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    r2_score
)

In [9]:
K_B = 8.617333262145e-5


def log_voltage_feature(voltage_v):
    voltage = np.asarray(voltage_v, dtype=float)
    log_voltage = np.zeros_like(voltage, dtype=float)
    positive_voltage = voltage > 0
    log_voltage[positive_voltage] = np.log(
        voltage[positive_voltage]
    )
    return log_voltage


def prepare_multistress_features(df):

    df = df.copy()

    temperature_kelvin = (
        df["Stress_Temperature_C"] + 273.15
    )

    df["Inverse_Temperature"] = (
        1 / temperature_kelvin
    )

    df["Log_Voltage"] = log_voltage_feature(
        df["Stress_Voltage_V"]
    )

    df["Humidity_Fraction"] = (
        df["Humidity_Percent"] / 100
    )

    return df


def train_multistress_reliability_model(df):

    df = prepare_multistress_features(df)

    failure_data = df[
        df["Censored"] == 0
    ].copy()

    x = failure_data[
        [
            "Inverse_Temperature",
            "Log_Voltage",
            "Humidity_Fraction"
        ]
    ]

    y = np.log(
        failure_data[
            "Failure_Time_Hours"
        ]
    )

    valid_rows = np.isfinite(x).all(axis=1) & np.isfinite(y)
    x = x.loc[valid_rows]
    y = y.loc[valid_rows]

    model = LinearRegression()

    model.fit(x,y)

    predictions = model.predict(x)

    mae = mean_absolute_error(
        y,
        predictions
    )

    r2 = r2_score(
        y,
        predictions
    )

    print("\nMULTI-STRESS MODEL")
    print("-" * 40)

    print(
        f"R² Score : {r2:.4f}"
    )

    print(
        f"MAE : {mae:.4f}"
    )

    print("\nMODEL COEFFICIENTS")

    for feature, coefficient in zip(
        x.columns,
        model.coef_
    ):

        print(
            f"{feature:<25}"
            f"{coefficient:>12.6f}"
        )

    return model


def predict_lifetime(
    model,
    temperature_c,
    voltage_v,
    humidity_percent
):

    temperature_kelvin = (
        temperature_c + 273.15
    )

    inverse_temperature = (
        1 / temperature_kelvin
    )

    log_voltage = log_voltage_feature(
        voltage_v
    )

    humidity_fraction = (
        humidity_percent / 100
    )

    X_new = np.array(
        [
            [
                inverse_temperature,
                log_voltage,
                humidity_fraction
            ]
        ]
    )

    predicted_log_life = (
        model.predict(X_new)[0]
    )

    predicted_lifetime = np.exp(
        predicted_log_life
    )

    return predicted_lifetime


def reliability_risk_category(
    predicted_lifetime
):

    if predicted_lifetime >= 3000:

        return "LOW RISK"

    elif predicted_lifetime >= 1500:

        return "MEDIUM RISK"

    else:

        return "HIGH RISK"


def run_multistress_prediction_engine():

    df = pd.read_csv(
        "reliability_dataset1.csv"
    )

    model = (
        train_multistress_reliability_model(
            df
        )
    )

    predicted_lifetime = (
        predict_lifetime(
            model,
            temperature_c=110,
            voltage_v=4.2,
            humidity_percent=60
        )
    )

    risk = (
        reliability_risk_category(
            predicted_lifetime
        )
    )

    print("\nPREDICTION RESULT")
    print("-" * 40)

    print(
        f"Predicted Lifetime : "
        f"{predicted_lifetime:.2f} hours"
    )

    print(
        f"Risk Category : "
        f"{risk}"
    )


run_multistress_prediction_engine()


MULTI-STRESS MODEL
----------------------------------------
R² Score : 0.0535
MAE : 0.6000

MODEL COEFFICIENTS
Inverse_Temperature        381.325188
Log_Voltage                  0.195692
Humidity_Fraction            1.298786

PREDICTION RESULT
----------------------------------------
Predicted Lifetime : 685.19 hours
Risk Category : HIGH RISK


c:\Users\anami\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
